# เข้าใจข้อมูลภาพ (Understanding Image Data)

**หัวข้อหลัก 1/4 ของ "ภาพรวม Image Processing"**

คอมพิวเตอร์มองเห็นภาพเป็น**ตัวเลข** ไม่ใช่รูปแบบที่คนมองเห็น — เรียนรู้ Pixel, Channel และ dtype
ผ่านภาพแผงวงจรจริง ภาพเดียวตลอดทั้งบท เพื่อให้เห็นความต่อเนื่องของแต่ละขั้นตอน


> เนื้อหาและภาพตัวอย่างอ้างอิงจากสไลด์ **COMPUTER VISIONS** โดย Asst.Prof.Dr. Amnach Khawne, King Mongkut's Institute of Technology Ladkrabang


## วิธีใช้ Notebook นี้

- รันทีละ Cell จากบนลงล่างด้วย **Shift + Enter**
- แต่ละหัวข้อแบ่งเป็น Cell ย่อยหลาย Cell ทำทีละขั้นตอน เพื่อให้เห็นผลลัพธ์ทันทีทีละ Cell
- ลองแก้ค่าตัวเลข (parameter) แล้วรันซ้ำ เพื่อดูว่าภาพเปลี่ยนไปอย่างไร
- **ต้องอัปโหลด `data.zip` ก่อน** (Cell ที่ 2) ทุกครั้งที่เปิด Notebook ใหม่ — Colab ลบไฟล์ทิ้งเมื่อ Runtime ถูกรีเซ็ต


## 0. เตรียมเครื่องมือ (Setup)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow  # แสดงภาพใน Colab แทน cv2.imshow()

print("OpenCV:", cv2.__version__)
print("NumPy :", np.__version__)


In [ ]:
def show_images(images, titles, cmap=None, figsize=(15, 5)):
    """แสดงภาพหลายภาพเรียงกันในแถวเดียว สำหรับเปรียบเทียบก่อน-หลัง"""
    n = len(images)
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        plt.subplot(1, n, i + 1)
        if img.ndim == 2:
            plt.imshow(img, cmap=cmap or "gray", vmin=0, vmax=255)
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))  # BGR -> RGB สำหรับ matplotlib
        plt.title(title, fontsize=11)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


## 1. เตรียมชุดข้อมูล (Dataset) — อัปโหลด `data.zip`

อัปโหลดไฟล์ **`data.zip`** ที่ได้รับจากผู้สอน (ภาพชุดเดียวกับที่ใช้ในสไลด์ COMPUTER VISIONS)
เมื่อรัน Cell ด้านล่างจะมีปุ่มให้เลือกไฟล์จากเครื่อง — เลือก `data.zip` แล้วรอจนแตกไฟล์เสร็จ

In [ ]:
import os
import zipfile

from google.colab import files

%cd /content
print("เลือกไฟล์ data.zip (ชุดภาพตัวอย่างจากสไลด์ COMPUTER VISIONS)")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall("/content/")

DATA_DIR = "/content/data"
print(f"แตกไฟล์ {zip_filename} เรียบร้อยแล้ว")
print("ไฟล์ภาพที่มีในโฟลเดอร์ data/:")
for fname in sorted(os.listdir(DATA_DIR)):
    print(" -", fname)


## 2. โหลดภาพแผงวงจร (pcb.jpg) — ภาพเดียวที่ใช้ตลอดบทนี้

In [ ]:
img_pcb = cv2.imread(f"{DATA_DIR}/pcb.jpg")
print("shape:", img_pcb.shape, " dtype:", img_pcb.dtype)


In [ ]:
cv2_imshow(img_pcb)


## 3. ภาพคือ NumPy Array

`img.shape` = **(H, W, C)** = สูง, กว้าง, จำนวน channel สี
**ข้อควรจำ:** NumPy อ่านแถว (y) ก่อนคอลัมน์ (x) เสมอ → indexing เขียนเป็น `img[y, x]`

อ้างอิง: [NumPy — Indexing basics](https://numpy.org/doc/stable/user/basics.indexing.html)

In [ ]:
H, W, C = img_pcb.shape
print(f"สูง (H) = {H}, กว้าง (W) = {W}, จำนวน channel (C) = {C}")


In [ ]:
x, y = 100, 50
pixel = img_pcb[y, x]
print(f"พิกเซลที่ (x={x}, y={y}) -> (B,G,R) = {pixel}")
print("เฉพาะ Blue channel (index 0):", img_pcb[y, x, 0])


## 4. ซูมดูพิกเซลทีละจุด (Zoom-in)

ภาพจริง ๆ คือ "ตาราง" ของตัวเลข ไม่ใช่เส้นโค้งต่อเนื่อง — ลองตัด (crop) ส่วนเล็ก ๆ ของ `pcb.jpg`
บริเวณขอบชิ้นส่วนบนแผงวงจร (มีรอยต่อของสี ทำให้เห็นความต่างของแต่ละพิกเซลชัดเจน)

In [ ]:
patch = img_pcb[60:72, 60:72]   # ครอบ 12x12 พิกเซล บริเวณขอบชิ้นส่วน
print("patch shape:", patch.shape)
cv2_imshow(patch)   # เล็กมาก แทบมองไม่เห็นรายละเอียด


In [ ]:
patch_zoom = cv2.resize(patch, None, fx=20, fy=20, interpolation=cv2.INTER_NEAREST)
cv2_imshow(patch_zoom)   # ขยาย 20 เท่า -> เห็นแต่ละพิกเซลเป็นสี่เหลี่ยมชัดเจน


## 5. ภาพสี = 3 Channel (B, G, R)

**สำคัญ:** OpenCV เก็บภาพสีเป็นลำดับ **BGR** แยกแต่ละ channel ด้วย `cv2.split()`
แต่ละ channel ที่แยกออกมาคือภาพ**ระดับเทา 1 ชั้น** (ยังไม่มีสี) — ดูให้เห็นก่อน แล้วค่อยใส่สีทีหลัง

ตรงกับสไลด์ 24-25 "การแยกและรวมช่องสีใน OpenCV"

อ้างอิง: [OpenCV — Changing Colorspaces](https://docs.opencv.org/4.13.0/df/d9d/tutorial_py_colorspaces.html)

In [ ]:
b, g, r = cv2.split(img_pcb)
print("แต่ละ channel shape:", b.shape, "(1 มิติ ไม่มีสี)")

show_images([b, g, r], ["B channel (grayscale)", "G channel (grayscale)", "R channel (grayscale)"], cmap="gray")


In [ ]:
zero = np.zeros_like(b)
b_vis = cv2.merge([b, zero, zero])   # ใส่ค่ากลับไปเฉพาะตำแหน่ง Blue
g_vis = cv2.merge([zero, g, zero])   # ใส่ค่ากลับไปเฉพาะตำแหน่ง Green
r_vis = cv2.merge([zero, zero, r])   # ใส่ค่ากลับไปเฉพาะตำแหน่ง Red

show_images([b_vis, g_vis, r_vis], ["Blue channel", "Green channel", "Red channel"])


In [ ]:
merged_back = cv2.merge([b, g, r])
print("รวมกลับแล้วเหมือนต้นฉบับหรือไม่:", np.array_equal(img_pcb, merged_back))
cv2_imshow(merged_back)

## 6. BGR vs RGB vs Grayscale — `cv2.cvtColor()`

matplotlib ต้องการภาพแบบ **RGB** แต่ OpenCV เก็บเป็น **BGR** — ถ้าไม่แปลงก่อน สีจะสลับ (แดง↔น้ำเงิน)
ตรงกับสไลด์ 25 "สลับ R/B: สีเพี้ยน"

In [ ]:
img_rgb = cv2.cvtColor(img_pcb, cv2.COLOR_BGR2RGB)

# เปรียบเทียบตรง ๆ ด้วย plt.imshow() (ไม่ผ่าน show_images ซึ่งแปลง BGR->RGB ให้อัตโนมัติอยู่แล้ว)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(img_pcb); plt.title("Raw BGR fed to plt -> wrong colors"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(img_rgb); plt.title("After COLOR_BGR2RGB -> correct colors"); plt.axis("off")
plt.show()


In [ ]:
img_gray = cv2.cvtColor(img_pcb, cv2.COLOR_BGR2GRAY)
print("shape สี:", img_pcb.shape, " shape เทา:", img_gray.shape, "(ไม่มี channel สีแล้ว)")

cv2_imshow(img_gray)


## 7. dtype: uint8 (0–255) และปัญหา Overflow

ภาพส่วนใหญ่เก็บค่าเป็น `uint8` (0–255) การบวกด้วย NumPy ตรง ๆ อาจ "วนกลับ" แต่ `cv2.add()` จะอิ่มตัวที่ 255 แทน

อ้างอิง: [NumPy — Data types](https://numpy.org/doc/stable/user/basics.types.html)

In [ ]:
pixel = np.uint8(250)
numpy_result = pixel + np.uint8(10)
cv2_result = cv2.add(np.array([[pixel]], dtype=np.uint8), np.array([[10]], dtype=np.uint8))

print("250 + 10 แบบ NumPy ตรง ๆ:", numpy_result, "(ผิด — วนกลับ)")
print("250 + 10 แบบ cv2.add() :", cv2_result[0, 0], "(ถูกต้อง — อิ่มตัวที่ 255)")


In [ ]:
# ลองบวกค่าความสว่าง 100 เข้าไปในภาพ img_pcb เพื่อดูผลลัพธ์ที่เกิด Overflow
add_val = np.uint8(100)

# 1. บวกแบบ NumPy ตรง ๆ (ภาพจะสีเพี้ยน เกิดจุดด่างดำ เพราะพิกเซลที่สว่างเกิน 255 จะวนกลับไปเป็นสีมืด)
img_numpy_add = img_pcb + add_val

# 2. บวกแบบ cv2.add (ภาพจะสว่างขึ้นอย่างถูกต้อง สีไม่ออกมาด่าง เพราะถูกล็อกสูงสุดไว้ที่ 255)
# ใช้ np.full ให้อยู่ในรูป array ขอบเขตเท่ากับภาพ
img_cv2_add = cv2.add(img_pcb, np.full(img_pcb.shape, 100, dtype=np.uint8))

# แสดงผลเปรียบเทียบด้วยฟังก์ชัน show_images (ที่ประกาศไว้ในหัวข้อที่ 0)
show_images([img_pcb, img_numpy_add, img_cv2_add],
            ["Original", "NumPy (+100) : Overflow", "cv2.add (+100) : Correct"])

## สรุปสิ่งที่เรียนในบทนี้

| แนวคิด | คำสั่งที่ใช้ |
|---|---|
| อ่าน/แสดงภาพ | `cv2.imread()`, `cv2_imshow()` |
| ดู/เข้าถึงพิกเซล | `img.shape`, `img[y, x]`, NumPy slicing |
| แยก/รวม channel | `cv2.split()`, `cv2.merge()` |
| แปลงสี | `cv2.cvtColor()` — `BGR2RGB`, `BGR2GRAY` |
| ป้องกัน overflow | `cv2.add()` แทนการบวก NumPy ตรง ๆ |

**ถัดไป:**  ปรับคุณภาพภาพ (Brightness / Contrast / Noise / Filter)


## แบบฝึกหัดท้ายบท (ลองทำเอง)

1. เปลี่ยนพิกัด `x, y` ในหัวข้อ 3 เป็นตำแหน่งอื่น แล้วดูค่าพิกเซล (B,G,R)
2. เปลี่ยนตำแหน่ง patch ในหัวข้อ 4 ไปครอบชิ้นส่วนอื่นบนแผงวงจร
3. ลองสร้างภาพสีเทียมที่เหลือเฉพาะ Green + Red (ไม่มี Blue) ในหัวข้อ 5 ดูว่าได้สีอะไร
